# Web App Data: COE Planner

This notebook prepares the data for a planned **COE planner web app** for potential car buyers (a Next.js front end hosted on Vercel). It runs [`analysis.ipynb`](analysis.ipynb) and saves the results the app needs to one JSON file, `outputs/coe_planner_data.json`. The app only displays these pre-computed numbers, so it needs no Python.

**When new COE results are published:** update the cleaned data, then re-run this notebook. It re-runs the analysis, so the app and the analysis always use the same numbers.

## App features and where their numbers come from

| App feature | Analysis section | JSON key |
|---|---|---|
| **"How much should I budget?"** Today's premium plus a buffer for bidding 1, 3, 6 or 12 months ahead, with a safety slider: usually enough (75%), very likely enough (90%), almost always enough (95%) | Section 2 | `premium_change_percentiles` and `latest` |
| **Quota signal.** When the next period's quota is announced, show its change and how premiums moved after similar changes in the past | Section 3 | `quota_bands` and `quota_slope` |
| **Category A vs B saving.** The current gap and the long-run median | Section 5 | `a_vs_b_gap` |
| **Renew or replace at year 10.** Today's renewal price (PQP) vs a new COE | Section 6 | `latest` (premium and PQP) |

**How the app should present the numbers:**
- Show **ranges from past data, not price forecasts**, and say so on the page.
- Show the date of the latest data (`data_to`).
- State that the figures cover the COE premium only, not the car's price, taxes, dealer packages or running costs.

In [1]:
# Run the analysis notebook without showing its charts, so its results are available here
import warnings
warnings.filterwarnings('ignore', message='.*non-interactive.*')
%matplotlib agg
%run ./analysis.ipynb

Partial periods:
quota_period  exercises
      2009Q4          2
      2020Q1          4
      2020Q2          2
      2026Q3          4


Period-to-period changes per category: 62


In [2]:
# Export the results a planning web app would need, as one JSON file
import json, os

OUT_DIR = '../outputs'
os.makedirs(OUT_DIR, exist_ok=True)
pcts = changes.groupby(['vehicle_class', 'months'])['change'].quantile([0.05, 0.5, 0.75, 0.9, 0.95]).unstack()
app_data = {
    'data_to': coe['month'].max(),
    'latest': {cat: {'premium': float(r['premium_f']), 'pqp': round(float(r['pqp']))} for cat, r in latest_rows.iterrows()},
    'premium_change_percentiles': {cat: {int(m): {f'p{int(q * 100)}': round(float(v), 4) for q, v in pcts.loc[(cat, m)].items()}
                                         for m in HORIZONS} for cat in CATS},
    'quota_bands': {cat: {band: {'periods': int(r['periods']), 'median_premium_change': round(float(r['median premium change']), 4),
                                 'share_premium_up': round(float(r['% premium went up']), 4)}
                          for band, r in bands.loc[cat].iterrows()} for cat in CATS},
    'quota_slope': {cat: round(float(reg.loc[cat, 'slope']), 4) for cat in CATS},
    'a_vs_b_gap': {'median_gap': float(ab['gap'].median()), 'latest_gap': float(ab['gap'].iloc[-1])},
}
with open(f'{OUT_DIR}/coe_planner_data.json', 'w') as f:
    json.dump(app_data, f, indent=2)
print('Saved', f'{OUT_DIR}/coe_planner_data.json')

Saved ../outputs/coe_planner_data.json


## Check the exported file

In [3]:
# Reload the file and show the budget buffer for Category A as the app would read it
with open(f'{OUT_DIR}/coe_planner_data.json') as f:
    saved = json.load(f)
print('Data to:', saved['data_to'])
print('Keys:', list(saved))
pd.DataFrame(saved['premium_change_percentiles']['Category A']).T.rename_axis('months ahead').style.format('{:+.1%}')

Data to: 2026-09
Keys: ['data_to', 'latest', 'premium_change_percentiles', 'quota_bands', 'quota_slope', 'a_vs_b_gap']


,p5,p50,p75,p90,p95
months ahead,,,,,
1,-14.2%,+1.5%,+6.0%,+11.7%,+15.1%
3,-18.9%,+3.4%,+12.1%,+20.1%,+26.5%
6,-20.6%,+6.7%,+18.0%,+31.6%,+39.3%
12,-29.3%,+10.7%,+30.5%,+53.8%,+69.5%
